# Setlist AI Composer

In [1]:
#Install libraries
import os
import json
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv()


True

In [2]:
# Authenticate to Google

# Full access for setlist agent (read + write)
SCOPES = [
    "https://www.googleapis.com/auth/documents",
    "https://www.googleapis.com/auth/drive"
]

def authenticate_google():
    creds = None

    # Load saved token if it exists
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)

    # If no valid credentials, run OAuth flow
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "credentials.json",
                SCOPES
            )
            creds = flow.run_local_server(port=0)

        # Save credentials for future runs
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    # Build API clients (this is what your agent will use later)
    docs_service = build("docs", "v1", credentials=creds)
    drive_service = build("drive", "v3", credentials=creds)

    return docs_service, drive_service


# Run authentication
docs_service, drive_service = authenticate_google()

print("Google authentication successful")


# Read Google Doc
def read_google_doc(docs_service, document_id):
    doc = docs_service.documents().get(documentId=document_id).execute()

    content = doc.get("body").get("content")

    text = ""

    for block in content:
        if "paragraph" in block:
            elements = block["paragraph"]["elements"]
            for el in elements:
                if "textRun" in el:
                    text += el["textRun"]["content"]

    return text


# Usage
DOCUMENT_ID = "10v3rXz9N8SJjl8_p_X6uuDSnYsrXQY8j2wHm8uQDz4E"

doc_text = read_google_doc(docs_service, DOCUMENT_ID)

print(doc_text)


# Parse the input data
def parse_setlist(text):
    lines = text.strip().split("\n")

    songs = []

    for line in lines:
        # skip headers or empty lines
        if "Setlist" in line or not line.strip():
            continue

        parts = [p.strip() for p in line.split("|")]

        if len(parts) == 3:
            song, artist, key = parts

            # clean key prefix if needed ("Key: G" → "G")
            if "Key:" in key:
                key = key.replace("Key:", "").strip()

            songs.append({
                "song": song,
                "artist": artist,
                "key": key
            })

    return songs


# test it
structured_setlist = parse_setlist(doc_text)

print(structured_setlist)



# Serialize setlist back into Google Doc format
def serialize_setlist(songs):
    lines = ["Setlist:\n"]

    for song in songs:
        line = f"{song['song']} | {song['artist']} | Key: {song['key']}"
        lines.append(line)

    return "\n".join(lines)


# Overwrite Google Doc
def overwrite_google_doc(docs_service, document_id, text):
    doc = docs_service.documents().get(documentId=document_id).execute()

    content = doc.get("body").get("content")
    end_index = content[-1]["endIndex"] - 1  # safe last position

    requests = [
        {
            "deleteContentRange": {
                "range": {
                    "startIndex": 1,
                    "endIndex": end_index
                }
            }
        },
        {
            "insertText": {
                "location": {
                    "index": 1
                },
                "text": text
            }
        }
    ]

    docs_service.documents().batchUpdate(
        documentId=document_id,
        body={"requests": requests}
    ).execute()


# 1. serialize structured state
new_doc_text = serialize_setlist(structured_setlist)

# 2. write back to Google Doc
#overwrite_google_doc(docs_service, DOCUMENT_ID, new_doc_text)

print("Doc updated successfully")

Google authentication successful
Setlist:

My Song | HER | Key: Am
Bound 2 | Kanye West | Key: G#
Mercy Me | Marvin Gaye | Key: E

[{'song': 'My Song', 'artist': 'HER', 'key': 'Am'}, {'song': 'Bound 2', 'artist': 'Kanye West', 'key': 'G#'}, {'song': 'Mercy Me', 'artist': 'Marvin Gaye', 'key': 'E'}]
Doc updated successfully


In [3]:
# Add Song
def add_song(songs, song, artist, key):
    songs.append({
        "song": song,
        "artist": artist,
        "key": key
    })
    return songs


# Remove song
def remove_song(songs, song_name):
    songs = [s for s in songs if s["song"].lower() != song_name.lower()]
    return songs


# Move Song
def move_song(songs, song_name, new_index):
    song_obj = None

    for s in songs:
        if s["song"].lower() == song_name.lower():
            song_obj = s
            break

    if song_obj:
        songs.remove(song_obj)
        songs.insert(new_index, song_obj)

    return songs



# Replace Song
def replace_song(songs, old_song, new_song, new_artist, new_key):
    for i, s in enumerate(songs):
        if s["song"].lower() == old_song.lower():
            songs[i] = {
                "song": new_song,
                "artist": new_artist,
                "key": new_key
            }
            break

    return songs


# Edit Key
def edit_key(songs, song_name, new_key):
    for s in songs:
        if s["song"].lower() == song_name.lower():
            s["key"] = new_key
            break

    return songs


# Delete All Songs
def delete_all_songs(songs):
    return []


#structured_setlist = add_song(structured_setlist, "Mercy Me", "Marvin Gaye", "E")
#structured_setlist = edit_key(structured_setlist, "Sugar", "Cm")
#structured_setlist = move_song(structured_setlist, "Ascension", 2)
#structured_setlist = remove_song(structured_setlist, "My Song")
#structured_setlist = replace_song(structured_setlist, "Sugar", "Real Love", "Mary J Blidge", "F#m")
#structured_setlist = delete_all_songs(structured_setlist)

new_doc_text = serialize_setlist(structured_setlist)
overwrite_google_doc(docs_service, DOCUMENT_ID, new_doc_text)
structured_setlist


[{'song': 'My Song', 'artist': 'HER', 'key': 'Am'},
 {'song': 'Bound 2', 'artist': 'Kanye West', 'key': 'G#'},
 {'song': 'Mercy Me', 'artist': 'Marvin Gaye', 'key': 'E'}]

In [15]:
# LLM

#Define tools

tools = [
    {
        "type": "function",
        "name": "add_song",
        "description": "Add a song to the setlist",
        "parameters": {
            "type": "object",
            "properties": {
                "song": {"type": "string"},
                "artist": {"type": "string"},
                "key": {"type": "string"}
            },
            "required": ["song", "artist", "key"]
        }
    },
    {
        "type": "function",
        "name": "remove_song",
        "description": "Remove a song from the setlist",
        "parameters": {
            "type": "object",
            "properties": {
                "song": {"type": "string"}
            },
            "required": ["song"]
        }
    },
    {
        "type": "function",
        "name": "move_song",
        "description": "Move a song to a position in the setlist",
        "parameters": {
            "type": "object",
            "properties": {
                "song": {"type": "string"},
                "new_index": {"type": "integer"}
            },
            "required": ["song", "new_index"]
        }
    },
    {
        "type": "function",
        "name": "edit_key",
        "description": "Change the musical key of a song",
        "parameters": {
            "type": "object",
            "properties": {
                "song": {"type": "string"},
                "key": {"type": "string"}
            },
            "required": ["song", "key"]
        }
    },
    {
        "type": "function",
        "name": "delete_all_songs",
        "description": "Remove all songs from the setlist",
        "parameters": {
            "type": "object",
            "properties": {}
        }
    },
    {
        "type": "function",
        "name": "replace_song",
        "description": "Replace a song entry completely",
        "parameters": {
            "type": "object",
            "properties": {
                "old_song": {"type": "string"},
                "new_song": {"type": "string"},
                "new_artist": {"type": "string"},
                "new_key": {"type": "string"}
            },
            "required": ["old_song", "new_song", "new_artist", "new_key"]
        }
    }
]

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def route_command(user_input):
    print("TOOLS BEING SENT:", tools)
    response = client.responses.create(
    model="gpt-5-mini",
    input=[
        {
            "role": "user",
            "content": user_input
        }
    ],
    tools=tools,
    tool_choice="auto"
)
    

    return response

In [16]:
def dispatch_tool(tool_name, args, songs):

    if tool_name == "add_song":
        return add_song(songs, **args)

    elif tool_name == "remove_song":
        return remove_song(songs, **args)

    elif tool_name == "move_song":
        return move_song(songs, **args)

    elif tool_name == "edit_key":
        return edit_key(songs, **args)

    elif tool_name == "replace_song":
        return replace_song(songs, **args)

    elif tool_name == "delete_all_songs":
        return delete_all_songs(songs)

    else:
        raise ValueError(f"Unknown tool: {tool_name}")

In [17]:
user_input = "Move Mercy Me to the top"

response = route_command(user_input)

tool_name = response.output[0].name
args = response.output[0].arguments

structured_setlist = dispatch_tool(
    tool_name,
    args,
    structured_setlist
)

new_doc_text = serialize_setlist(structured_setlist)

overwrite_google_doc(
    docs_service,
    DOCUMENT_ID,
    new_doc_text
)

TOOLS BEING SENT: [{'type': 'function', 'name': 'add_song', 'description': 'Add a song to the setlist', 'parameters': {'type': 'object', 'properties': {'song': {'type': 'string'}, 'artist': {'type': 'string'}, 'key': {'type': 'string'}}, 'required': ['song', 'artist', 'key']}}, {'type': 'function', 'name': 'remove_song', 'description': 'Remove a song from the setlist', 'parameters': {'type': 'object', 'properties': {'song': {'type': 'string'}}, 'required': ['song']}}, {'type': 'function', 'name': 'move_song', 'description': 'Move a song to a position in the setlist', 'parameters': {'type': 'object', 'properties': {'song': {'type': 'string'}, 'new_index': {'type': 'integer'}}, 'required': ['song', 'new_index']}}, {'type': 'function', 'name': 'edit_key', 'description': 'Change the musical key of a song', 'parameters': {'type': 'object', 'properties': {'song': {'type': 'string'}, 'key': {'type': 'string'}}, 'required': ['song', 'key']}}, {'type': 'function', 'name': 'delete_all_songs', 'd

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [9]:
import json

print(json.dumps(tools[0], indent=2))

{
  "type": "function",
  "function": {
    "name": "add_song",
    "description": "Add a song to the setlist",
    "parameters": {
      "type": "object",
      "properties": {
        "song": {
          "type": "string"
        },
        "artist": {
          "type": "string"
        },
        "key": {
          "type": "string"
        }
      },
      "required": [
        "song",
        "artist",
        "key"
      ]
    }
  }
}


In [ ]:
'''
User request
   ↓
LLM (decides action)
   ↓
Tool dispatcher (your functions)
   ↓
State update (Python list)
   ↓
Serialize
   ↓
Google Doc update
'''